In [1]:
import requests
import pandas as pd
import time

In [2]:
subjects = [
    "self_help",
    "psychology",
    "motivation",
    "mindfulness",
    "habits",
    "leadership",
    "communication",
    "creativity",
    "productivity",
    "emotions"
]

In [3]:
def get_subject_works(subject, limit=50):
    url = f"https://openlibrary.org/subjects/{subject}.json"
    
    params = {"limit": limit}
    res = requests.get(url, params=params)
    
    if res.status_code != 200:
        return []
    
    data = res.json()
    
    works = []
    
    for w in data.get("works", []):
        works.append({
            "title": w.get("title", ""),
            "work_key": w.get("key", ""),
            "subject": subject,
            "source": "openlibrary_api"
        })
    
    return works

In [4]:
all_books = []

for s in subjects:
    print("Fetching:", s)
    
    books = get_subject_works(s, limit=100)
    all_books.extend(books)
    
    time.sleep(1)

df_api = pd.DataFrame(all_books)

df_api.shape

Fetching: self_help
Fetching: psychology
Fetching: motivation
Fetching: mindfulness
Fetching: habits
Fetching: leadership
Fetching: communication
Fetching: creativity
Fetching: productivity
Fetching: emotions


(969, 4)

In [5]:
def get_work_details(work_key):
    url = f"https://openlibrary.org{work_key}.json"
    
    res = requests.get(url)
    
    if res.status_code != 200:
        return ""
    
    data = res.json()
    
    desc = data.get("description", "")
    
    if isinstance(desc, dict):
        desc = desc.get("value", "")
    
    return desc

In [6]:
descriptions = []

for i, row in df_api.head(100).iterrows():
    desc = get_work_details(row["work_key"])
    descriptions.append(desc)
    
    time.sleep(0.5)

df_api.loc[:99, "description"] = descriptions

In [7]:
df_api["description"] = df_api["description"].fillna("")

In [8]:
df_api["content"] = (
    df_api["title"] + " " +
    df_api["subject"] + " " +
    df_api["description"]
)

In [9]:
df_api.head()

,title,work_key,subject,source,description,content
0,The Monk Who Sold His Ferrari,/works/OL276492W,self_help,openlibrary_api,Includes a bonus excerpt of Robin Sharma's upc...,The Monk Who Sold His Ferrari self_help Includ...
1,The Secret,/works/OL15839737W,self_help,openlibrary_api,Fragments of a Great Secret have been found in...,The Secret self_help Fragments of a Great Secr...
2,The Power of Focused Thinking,/works/OL675386W,self_help,openlibrary_api,"Ben shu fen wei qi ge bu fen, Fen wei bai se s...",The Power of Focused Thinking self_help Ben sh...
3,嫌われる勇気,/works/OL19744000W,self_help,openlibrary_api,"*""The Courage to Be Disliked,* already an enor...","嫌われる勇気 self_help *""The Courage to Be Disliked,..."
4,The Definitive Book of Body Language,/works/OL5599445W,self_help,openlibrary_api,This book isolates and examines each component...,The Definitive Book of Body Language self_help...


In [10]:
df_api.shape

(969, 6)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

tfidf_matrix = tfidf.fit_transform(df_api["content"])

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)